# Qwen3.6-27B-FP8 — raw OCR + preprocessing experiment

**One question:** can this model read your poor-quality scanned Algerian KYC documents?

No KYC schema. No field extraction. No database. No cross-document matching. No MRZ
reconstruction. No task planner. **One page → one image → one Qwen call → raw text.**

## Architecture

```
PDF ──pymupdf──► render page at RENDER_DPI  ──────────────► ORIGINAL (never modified, always kept)
                          │
                          ├─► quality analysis (blur, contrast, brightness, ink)   [CPU, ~20 ms]
                          │
                          └─► conservative preprocessing, each step optional       [CPU]
                                 orientation → deskew → grayscale → contrast
                                 → denoise → sharpen → (threshold, OFF) → upscale
                                          │
                                          ▼
                                   PREPROCESSED  ──► ONE Qwen call ──► raw text
```

Both images are saved when `SAVE_DEBUG_IMAGES = True`, so you can open the PDF page, the image
that was actually sent, and the transcription side by side. That comparison is the whole point.

## Design decisions that matter

**`bfloat16`, never `float16`.** A VL vision tower overflows fp16's 65504 ceiling; the result is
NaN logits, and `argmax` over an all-NaN row returns index 0 — which in Qwen-family byte-level BPE
decodes to `"!"`. That is the mechanism behind a `!!!!!!` output. bf16 has fp32's exponent range.
Section 16 tests for it explicitly before any document is touched.

**FP8 excludes the vision tower.** Per-block FP8 scales on a patch embedding can underflow to
zero and divide by zero. The encoder is a small share of the weights and is what resolves small
glyphs, so it stays in bf16.

**Preprocessing is conservative by default.** Aggressive binarisation destroys thin strokes, MRZ
characters, accents and handwriting — exactly what these documents are made of. `USE_THRESHOLD`
is **off**. Contrast and denoise fire only when the measured quality says they should.

**One call per page.** 8 pages = 8 calls. `COMPARE_PREPROCESSING` can run original-vs-preprocessed
on **one** page if you ask for it; it never runs on every page.

## How to run

1. Leave `DRY_RUN = True`. You get the customer, the PDFs, real page counts, the planned call
   count and the preprocessing configuration — with no GPU work.
2. Set `DRY_RUN = False` to run the experiment.
3. Read `04_debug/` next to `01_raw_ocr/` and judge the transcriptions yourself.

In [ ]:
# =========================================================================
# CELL 4 — INSTALLATION (pinned)
# =========================================================================
# Domino images normally ship torch/transformers/accelerate. Reinstalling torch can break the
# CUDA build, so nothing runs unless INSTALL_MISSING is set explicitly.
#
# Versions chosen for Qwen-VL-family multimodal inference with FP8 on an H100:
#   torch>=2.3,<2.8              SDPA fused attention; FP8 storage support
#   transformers>=4.49,<5        AutoModelForImageTextToText + FineGrainedFP8Config
#   accelerate>=0.30             device_map / max_memory
#   tokenizers>=0.19             matches the transformers range
#   safetensors>=0.4.3           FP8 tensor loading
#   pillow>=10.0                 image handling
#   pymupdf>=1.24                PDF render + clip  (import pymupdf, NOT fitz)
#   numpy>=1.24  pandas>=2.0
#   opencv-python-headless>=4.9  preprocessing (deskew, CLAHE, denoise)
#
# NEVER installed, NEVER imported: flash_attn.
INSTALL_MISSING = False

PINS = {
    "torch": ">=2.3,<2.8", "transformers": ">=4.49,<5", "accelerate": ">=0.30",
    "tokenizers": ">=0.19", "safetensors": ">=0.4.3", "pillow": ">=10.0",
    "pymupdf": ">=1.24", "numpy": ">=1.24", "pandas": ">=2.0",
    "opencv-python-headless": ">=4.9",
}
SAFE_TO_INSTALL = ["pymupdf", "opencv-python-headless", "pillow", "pandas", "numpy"]

print("pinned targets:")
for k, v in PINS.items():
    print(f"  pip install '{k}{v}'" + ("" if k in SAFE_TO_INSTALL else "   # already in the image"))

if INSTALL_MISSING:
    import subprocess, sys
    for pkg in SAFE_TO_INSTALL:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{pkg}{PINS[pkg]}"],
                       check=False)
    print("\ninstall attempted -- RESTART THE KERNEL before continuing")
else:
    print("\nINSTALL_MISSING = False (recommended)")

In [ ]:
# =========================================================================
# CELL 5 / 6 — IMPORTS AND VERSION VERIFICATION
# =========================================================================
from __future__ import annotations

import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import io, re, sys, json, math, time, random, logging, platform, subprocess, traceback
import importlib, inspect
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

Image.MAX_IMAGE_PIXELS = None

# PyMuPDF under its current import name. The deprecated `fitz` alias is not used anywhere.
try:
    import pymupdf
except Exception as exc:
    pymupdf = None
    print("pymupdf import FAILED:", exc)

try:
    import cv2
except Exception:
    cv2 = None

try:
    import torch
except Exception:
    torch = None

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("ocr")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
utcnow = lambda: datetime.now(timezone.utc).isoformat()

print(f"{'python':<24}: {platform.python_version()}")
print(f"{'platform':<24}: {platform.platform()}")
for name in ("torch", "torchvision", "transformers", "accelerate", "tokenizers", "safetensors",
             "numpy", "pandas", "PIL", "pymupdf", "cv2"):
    try:
        m = importlib.import_module(name)
        print(f"{name:<24}: {getattr(m, '__version__', 'present')}")
    except Exception:
        print(f"{name:<24}: NOT INSTALLED")
if torch is not None:
    print(f"{'torch CUDA build':<24}: {torch.version.cuda}")
    print(f"{'cuda available':<24}: {torch.cuda.is_available()}")
try:
    print(f"{'flash_attn present':<24}: "
          f"{importlib.util.find_spec('flash_attn') is not None}  (never imported or used)")
except Exception:
    pass
print(f"{'run id':<24}: {RUN_ID}")

In [ ]:
# =========================================================================
# CELL 7 — CONFIGURATION  (everything tunable lives here)
# =========================================================================
@dataclass
class Config:
    # ---- model / data paths ---------------------------------------------
    MODEL_PATH: str = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main"
    CUSTOMER_ROOT: Path = Path("/domino/datasets/local/kyc/customers")
    OUTPUT_DIR: Path = Path("/mnt/output_qwen36_raw_ocr")

    # ---- experiment scope -------------------------------------------------
    DRY_RUN: bool = True                  # True = plan only, no Qwen call can execute
    RANDOM_SEED: int = 42
    CUSTOMER_ID: Optional[str] = None     # None = pick one folder at random with the seed
    COMPARE_PREPROCESSING: bool = False   # if True: ONE page is run original vs preprocessed
    MAX_RETRIES: int = 0                  # baseline: no retries

    # ---- rendering --------------------------------------------------------
    RENDER_DPI: int = 300                 # high render; the model input is capped below
    MAX_IMAGE_DIMENSION: int = 1800       # longest side sent to the model
    MAX_PAGES_PER_PDF: int = 50           # runaway guard only; real counts come from the PDF

    # ---- preprocessing switches (conservative by default) ------------------
    USE_AUTO_ROTATION: bool = True        # 90/180/270 only, and only on a confident margin
    USE_DESKEW: bool = True               # small angles only
    USE_GRAYSCALE: bool = True
    USE_CONTRAST: bool = True             # CLAHE, applied only when contrast measures low
    USE_BRIGHTNESS: bool = True           # flatten uneven illumination when measured
    USE_DENOISE: bool = True              # edge-preserving only, when noise measures high
    USE_SHARPEN: bool = True              # mild unsharp mask
    USE_THRESHOLD: bool = False           # OFF: binarisation destroys thin strokes, MRZ, accents
    USE_BORDER_CLEANUP: bool = True       # crop uniform scanner borders
    USE_PERSPECTIVE: bool = False         # OFF by default: only when a quad is clearly detected
    UPSCALE_FACTOR: float = 1.0           # 1.0 = off; raised automatically if glyphs are tiny
    MIN_TEXT_HEIGHT_PX: float = 14.0      # below this in the sent image, upscale
    DESKEW_MAX_DEG: float = 12.0
    ROTATION_MIN_MARGIN: float = 0.08     # below this the rotation heuristic abstains

    # ---- generation -------------------------------------------------------
    MAX_NEW_TOKENS: int = 512             # a page transcription; not an essay
    DO_SAMPLE: bool = False
    TEMPERATURE: float = 0.0
    REPETITION_PENALTY: float = 1.0       # MUST be 1.0: >1 corrupts repeats and "<<<"
    NO_REPEAT_NGRAM_SIZE: int = 0         # MUST be 0: would forbid "<<" in an MRZ
    DISABLE_THINKING: bool = True

    # ---- model loading ----------------------------------------------------
    TORCH_DTYPE: str = "bfloat16"         # NOT float16 -- see the note in CELL 1
    DEVICE_MAP: str = "auto"
    ATTENTION_BACKEND: str = "sdpa"
    USE_FP8: bool = True
    FP8_SKIP_MODULES: Tuple[str, ...] = ("lm_head", "visual", "vision_tower", "vision_model",
                                         "merger", "multi_modal_projector")
    MAX_VISUAL_TOKENS: int = 2048         # processor max_pixels: the prefill cost knob
    MIN_VISUAL_TOKENS: int = 256
    RESERVE_VRAM_GIB: float = 6.0
    ALLOW_CPU_OFFLOAD: bool = False       # False: fail loudly rather than run 50x slower

    # ---- safety / output --------------------------------------------------
    MAX_QWEN_CALLS: int = 60
    MAX_RUNTIME_SECONDS: float = 1800.0
    WARN_CALL_SECONDS: float = 45.0
    SAVE_DEBUG_IMAGES: bool = True        # True for this experiment: you need to SEE the input
    RUN_WARMUP: bool = True
    MOCK_MODEL: bool = False              # rehearse the notebook without a GPU

    def dirs(self) -> Dict[str, Path]:
        d = {k: self.OUTPUT_DIR / v for k, v in
             {"raw": "01_raw_ocr", "reports": "02_reports", "debug": "04_debug"}.items()}
        for p in d.values():
            p.mkdir(parents=True, exist_ok=True)
        return d


CFG = Config()
DIRS = CFG.dirs()
random.seed(CFG.RANDOM_SEED)

print(f"{'model':<26}: {CFG.MODEL_PATH}")
print(f"{'customers':<26}: {CFG.CUSTOMER_ROOT}")
print(f"{'output':<26}: {CFG.OUTPUT_DIR}")
print(f"{'DRY_RUN':<26}: {CFG.DRY_RUN}")
print(f"{'render / model input':<26}: {CFG.RENDER_DPI} dpi -> max {CFG.MAX_IMAGE_DIMENSION}px / "
      f"{CFG.MAX_VISUAL_TOKENS} visual tokens")
print(f"{'generation':<26}: max_new_tokens={CFG.MAX_NEW_TOKENS}, do_sample={CFG.DO_SAMPLE}")
print(f"{'dtype / attention':<26}: {CFG.TORCH_DTYPE} / {CFG.ATTENTION_BACKEND}")
print(f"{'preprocessing':<26}: rot={CFG.USE_AUTO_ROTATION} deskew={CFG.USE_DESKEW} "
      f"gray={CFG.USE_GRAYSCALE} contrast={CFG.USE_CONTRAST} denoise={CFG.USE_DENOISE} "
      f"sharpen={CFG.USE_SHARPEN} threshold={CFG.USE_THRESHOLD}")

In [ ]:
# =========================================================================
# CELL 8 — GPU DIAGNOSTICS (before the model)
# =========================================================================
# Three DIFFERENT measurements, routinely confused:
#   nvidia-smi used/free    the whole card, all processes
#   torch.memory_allocated  tensors this process currently holds
#   torch.memory_reserved   the caching allocator's pool for this process
# torch allocated == 0 before the model loads is CORRECT, not a failure.
MEMORY_TRACE: List[Dict[str, Any]] = []


def gpu_memory_snapshot(tag: str, verbose: bool = False) -> Dict[str, Any]:
    rec: Dict[str, Any] = {"tag": tag, "timestamp": utcnow(), "pid": os.getpid()}
    if torch is not None and torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        rec.update({
            "torch_allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3),
            "torch_reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 3),
            "torch_peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3),
            "torch_peak_reserved_gib": round(torch.cuda.max_memory_reserved() / 2**30, 3),
            "cuda_free_gib": round(free / 2**30, 2),
            "cuda_total_gib": round(total / 2**30, 2),
            "cuda_device": torch.cuda.current_device()})
        try:
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used,memory.free,utilization.gpu",
                 "--format=csv,noheader,nounits"], capture_output=True, text=True,
                timeout=10).stdout.strip().split("\n")[0].split(",")
            rec.update({"smi_used_gib": round(int(out[0]) / 1024, 2),
                        "smi_free_gib": round(int(out[1]) / 1024, 2),
                        "smi_util_pct": int(out[2])})
        except Exception:
            pass
    MEMORY_TRACE.append(rec)
    if verbose:
        for k, v in rec.items():
            if k not in ("tag", "timestamp"):
                print(f"  {k:<26}: {v}")
    return rec


if torch is not None and torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"gpu[{i}]                     : {p.name}")
        print(f"{'total memory':<26}: {p.total_memory/2**30:.1f} GiB")
        print(f"{'compute capability':<26}: sm_{p.major}{p.minor}")
    print(f"{'torch CUDA':<26}: {torch.version.cuda}")
    print(f"{'current device':<26}: {torch.cuda.current_device()}")
    print(f"{'process pid':<26}: {os.getpid()}")
    print("\nMEMORY BEFORE MODEL LOAD:")
    gpu_memory_snapshot("before_model", verbose=True)
    print("\n  torch_allocated 0.0 here is EXPECTED -- nothing is loaded yet.")
    print("  If smi_free is far below cuda_total, ANOTHER PROCESS holds this card.")
else:
    print("no CUDA device visible -- set CFG.MOCK_MODEL = True to rehearse the notebook")
    gpu_memory_snapshot("before_model")

In [ ]:
# =========================================================================
# CELL 9 — LOCAL MODEL INSPECTION  (read the directory, assume nothing)
# =========================================================================
MODEL_DIR = Path(CFG.MODEL_PATH)
MODEL_CONFIG: Dict[str, Any] = {}
MODEL_OK = MODEL_DIR.exists()
print(f"{'model directory':<26}: {MODEL_DIR}")
print(f"{'exists':<26}: {MODEL_OK}")

if MODEL_OK:
    cfg_file = MODEL_DIR / "config.json"
    if cfg_file.exists():
        MODEL_CONFIG = json.loads(cfg_file.read_text())
        print(f"{'model_type':<26}: {MODEL_CONFIG.get('model_type')}")
        print(f"{'architectures':<26}: {MODEL_CONFIG.get('architectures')}")
        print(f"{'torch_dtype (config)':<26}: {MODEL_CONFIG.get('torch_dtype')}")
        print(f"{'quantization_config':<26}: "
              f"{json.dumps(MODEL_CONFIG.get('quantization_config'))}")
        vc = MODEL_CONFIG.get("vision_config") or {}
        tc = MODEL_CONFIG.get("text_config") or {}
        print(f"{'vision_config present':<26}: {bool(vc)}")
        if vc:
            print(f"{'  vision hidden/depth':<26}: {vc.get('hidden_size')} / "
                  f"{vc.get('depth') or vc.get('num_hidden_layers')}")
            print(f"{'  patch / merge size':<26}: {vc.get('patch_size')} / "
                  f"{vc.get('spatial_merge_size')}")
        if tc:
            print(f"{'  text hidden/layers':<26}: {tc.get('hidden_size')} / "
                  f"{tc.get('num_hidden_layers')}")
        print(f"{'files':<26}: "
              f"{sorted(p.name for p in MODEL_DIR.glob('*.json'))[:8]}")
        weights = sorted(MODEL_DIR.glob("*.safetensors"))
        print(f"{'weight shards':<26}: {len(weights)} safetensors "
              f"({sum(p.stat().st_size for p in weights)/2**30:.1f} GiB)")
    else:
        print("!! config.json not found in the model directory")
        MODEL_OK = False
else:
    siblings = []
    parent = MODEL_DIR.parent.parent if MODEL_DIR.name == "main" else MODEL_DIR.parent
    if parent.exists():
        siblings = sorted(p.name for p in parent.iterdir() if p.is_dir())[:12]
    print(f"!! not found. Directories under {parent}: {siblings}")
    print("   Set CFG.MODEL_PATH to the correct directory, or CFG.MOCK_MODEL = True.")

HAS_VISION = bool(MODEL_CONFIG.get("vision_config")) or \
    any(h in " ".join(MODEL_CONFIG.get("architectures") or []).lower()
        for h in ("vl", "vision", "imagetext"))
print(f"\n{'MULTIMODAL (vision tower)':<26}: {HAS_VISION}")
if MODEL_OK and not HAS_VISION:
    print("!! this checkpoint exposes no vision tower -- it cannot read images.")

In [ ]:
# =========================================================================
# CELL 10 — FP8 CONFIGURATION
# =========================================================================
# The checkpoint is named ...-FP8. Two cases, handled differently:
#   (a) config.json already declares quantization_config -> the checkpoint IS quantised. Use it
#       as stored. Do NOT build another config and do NOT call .half()/.float(), which would
#       bypass or destroy the quantisation.
#   (b) no declaration -> build FineGrainedFP8Config, excluding the vision tower.
#
# Why exclude the vision tower: per-block FP8 scales on a patch embedding can underflow to zero,
# giving a division by zero -> NaN logits -> argmax returns token id 0 -> "!!!!!!" output.
FP8 = {"path": None, "class_found": False, "config": None, "mode": None, "reason": ""}

ckpt_quant = (MODEL_CONFIG.get("quantization_config") or {}) if MODEL_CONFIG else {}
if not CFG.USE_FP8 or CFG.MOCK_MODEL:
    FP8["mode"] = "disabled"
    FP8["reason"] = "USE_FP8 False or MOCK_MODEL True"
elif ckpt_quant:
    FP8["mode"] = "checkpoint"
    FP8["reason"] = (f"checkpoint declares quant_method={ckpt_quant.get('quant_method')} -- "
                     "loading as stored, no manual conversion")
else:
    # The documented import path is tried first. Verified against the installed package: the
    # class actually lives in transformers.utils.quantization_config and is re-exported at the
    # transformers top level; transformers.integrations.finegrained_fp8 holds FP8Linear and the
    # quantiser, not the config class.
    for mod_name, attr in [("transformers.integrations.finegrained_fp8", "FineGrainedFP8Config"),
                           ("transformers", "FineGrainedFP8Config"),
                           ("transformers.utils.quantization_config", "FineGrainedFP8Config")]:
        try:
            cls = getattr(importlib.import_module(mod_name), attr)
            params = [p for p in inspect.signature(cls.__init__).parameters
                      if p not in ("self", "args", "kwargs")]
            kwargs: Dict[str, Any] = {}
            if "activation_scheme" in params:
                kwargs["activation_scheme"] = "dynamic"
            if "weight_block_size" in params:
                kwargs["weight_block_size"] = (128, 128)
            if "modules_to_not_convert" in params:
                kwargs["modules_to_not_convert"] = list(CFG.FP8_SKIP_MODULES)
            FP8.update({"path": f"{mod_name}.{attr}", "class_found": True,
                        "config": cls(**kwargs), "mode": "explicit", "params": params,
                        "kwargs": kwargs, "reason": "built for an unquantised checkpoint"})
            break
        except Exception as exc:
            FP8["reason"] = f"{type(exc).__name__}"

print(f"{'FP8 mode':<26}: {FP8['mode']}")
print(f"{'reason':<26}: {FP8['reason']}")
if FP8["class_found"]:
    print(f"{'resolved via':<26}: {FP8['path']}")
    print(f"{'constructor params':<26}: {FP8.get('params')}")
    print(f"{'configured with':<26}: {json.dumps(FP8.get('kwargs'), default=str)}")


def resolve_attention(cfg: Config = CFG) -> str:
    """SDPA is PyTorch's own fused attention: no extra package, works on H100.
    flash_attn is never imported. eager is the universal fallback."""
    req = (cfg.ATTENTION_BACKEND or "sdpa").lower()
    if req in ("flash_attention_2", "flash_attn"):
        log.warning("FlashAttention is not a dependency here; using sdpa")
        req = "sdpa"
    if req == "sdpa" and torch is not None and \
            not hasattr(torch.nn.functional, "scaled_dot_product_attention"):
        log.warning("this torch has no scaled_dot_product_attention -> eager")
        req = "eager"
    return req


ATTENTION_REQUESTED = resolve_attention(CFG)
print(f"\n{'attention requested':<26}: {ATTENTION_REQUESTED}  (confirmed after load in CELL 13)")

In [ ]:
# =========================================================================
# CELL 11 / 12 / 13 — PROCESSOR, MODEL, DEVICE VERIFICATION  (one instance each)
# =========================================================================
class MockEngine:
    is_mock, name, attn = True, "MOCK", "n/a"
    model_class, placement = "MockEngine", "MOCK"
    device_map_summary, quantization, param_devices = {}, {}, []
    load_seconds = processor_seconds = 0.0
    first_param_device = vision_device = "n/a"
    processor = tokenizer = None
    pixel_cap = False


class QwenOCREngine:
    _instance = None
    is_mock = False

    def __init__(self, cfg: Config = CFG):
        from transformers import AutoConfig, AutoProcessor
        import transformers as tf
        assert torch is not None, "PyTorch is required"
        assert MODEL_OK, f"model directory unusable: {cfg.MODEL_PATH}"
        self.cfg = cfg
        self.hf_config = AutoConfig.from_pretrained(cfg.MODEL_PATH, trust_remote_code=True,
                                                    local_files_only=True)
        archs = list(getattr(self.hf_config, "architectures", []) or [])

        # ---- processor: exactly one, from the SAME directory as the weights ----
        t0 = time.perf_counter()
        try:
            self.processor = AutoProcessor.from_pretrained(
                cfg.MODEL_PATH, trust_remote_code=True, local_files_only=True,
                min_pixels=cfg.MIN_VISUAL_TOKENS * 28 * 28,
                max_pixels=cfg.MAX_VISUAL_TOKENS * 28 * 28)
            self.pixel_cap = True
        except TypeError:
            self.processor = AutoProcessor.from_pretrained(cfg.MODEL_PATH, trust_remote_code=True,
                                                           local_files_only=True)
            self.pixel_cap = False
            log.warning("processor does not accept min/max_pixels: visual tokens are uncapped; "
                        "lower CFG.MAX_IMAGE_DIMENSION instead")
        self.processor_seconds = round(time.perf_counter() - t0, 2)
        self.tokenizer = getattr(self.processor, "tokenizer", self.processor)
        self.has_chat_template = bool(getattr(self.processor, "chat_template", None) or
                                      getattr(self.tokenizer, "chat_template", None))

        kwargs: Dict[str, Any] = dict(
            torch_dtype=getattr(torch, cfg.TORCH_DTYPE),      # bf16, never fp16
            device_map=cfg.DEVICE_MAP, trust_remote_code=True, local_files_only=True,
            low_cpu_mem_usage=True, attn_implementation=ATTENTION_REQUESTED)
        if FP8.get("config") is not None:
            kwargs["quantization_config"] = FP8["config"]
        if cfg.RESERVE_VRAM_GIB > 0 and torch.cuda.is_available() and cfg.DEVICE_MAP == "auto":
            mm = {}
            for i in range(torch.cuda.device_count()):
                _, total = torch.cuda.mem_get_info(i)
                mm[i] = f"{max(1, int(total / 2**30) - int(cfg.RESERVE_VRAM_GIB))}GiB"
            if cfg.ALLOW_CPU_OFFLOAD:
                mm["cpu"] = "64GiB"
            kwargs["max_memory"] = mm

        t0, last, self.model = time.perf_counter(), None, None
        for cls_name in archs + ["AutoModelForImageTextToText", "AutoModelForVision2Seq"]:
            if not hasattr(tf, cls_name):
                continue
            try:
                self.model = getattr(tf, cls_name).from_pretrained(cfg.MODEL_PATH, **kwargs)
                self.model_class = cls_name
                break
            except Exception as exc:
                last = f"{cls_name}: {type(exc).__name__}: {exc}"
                msg = str(exc).lower()
                if "quantization_config" in kwargs and "quantiz" in msg:
                    log.warning("checkpoint refused an explicit FP8 config; using its own")
                    kwargs.pop("quantization_config")
                elif "attention" in msg and kwargs.get("attn_implementation") != "eager":
                    log.warning("attn_implementation=%s refused; retrying with eager",
                                kwargs["attn_implementation"])
                    kwargs["attn_implementation"] = "eager"
        if self.model is None:
            raise RuntimeError(f"could not load the model. last error: {last}")
        self.load_seconds = round(time.perf_counter() - t0, 1)
        self.attn = kwargs.get("attn_implementation", ATTENTION_REQUESTED)
        self.model.eval()                                     # never training mode
        gc_ = getattr(self.model, "generation_config", None)
        if gc_ is not None:
            gc_.do_sample = cfg.DO_SAMPLE
            gc_.temperature = gc_.top_p = gc_.top_k = None

        # ---- device verification: evidence, not assumption ----
        dm = getattr(self.model, "hf_device_map", None) or {}
        summary = defaultdict(int)
        for _, dev in dm.items():
            summary[str(dev)] += 1
        self.device_map_summary = dict(summary)
        devs = {str(p.device) for _, p in self.model.named_parameters()}
        self.param_devices = sorted(devs)
        has_gpu = any("cuda" in d for d in devs)
        has_cpu = any(d == "cpu" for d in devs)
        self.placement = ("MIXED_CPU_GPU" if has_gpu and has_cpu else
                          "GPU_ONLY" if has_gpu else "CPU_ONLY" if has_cpu else "UNKNOWN")
        self.first_param_device = str(next(self.model.parameters()).device)
        self.vision_device = next((str(p.device) for n, p in self.model.named_parameters()
                                   if any(h in n for h in ("visual", "vision"))), "n/a")
        self.lm_device = next((str(p.device) for n, p in self.model.named_parameters()
                               if any(h in n for h in ("language_model", "model.layers"))), "n/a")
        qc = getattr(self.model.config, "quantization_config", None)
        method = getattr(qc, "quant_method", None) if qc is not None else None
        n_fp8 = sum(1 for _, m in self.model.named_modules() if type(m).__name__ == "FP8Linear")
        dtypes = defaultdict(int)
        for _, p in list(self.model.named_parameters())[:400]:
            dtypes[str(p.dtype)] += 1
        self.quantization = {"quant_method": str(getattr(method, "value", method)),
                             "n_fp8_linear_modules": n_fp8, "param_dtypes": dict(dtypes),
                             "fp8_active": bool(n_fp8 > 0 or
                                                (method and "fp8" in str(method).lower()))}
        self.name = f"{Path(cfg.MODEL_PATH).parent.name} [{self.model_class}]"

    @classmethod
    def get(cls, cfg: Config = CFG):
        if cls._instance is None:
            cls._instance = cls(cfg)
        return cls._instance


def load_engine(cfg: Config = CFG):
    """ONE model and ONE processor for the whole experiment."""
    return MockEngine() if cfg.MOCK_MODEL else QwenOCREngine.get(cfg)


ENGINE = load_engine(CFG)
snap_after = gpu_memory_snapshot("after_model")

print("=" * 64)
print("MODEL LOADED")
print("=" * 64)
print(f"{'MODEL CLASS':<26}: {ENGINE.model_class}")
print(f"{'processor class':<26}: {type(getattr(ENGINE, 'processor', None)).__name__}")
print(f"{'tokenizer class':<26}: {type(getattr(ENGINE, 'tokenizer', None)).__name__}")
print(f"{'chat template present':<26}: {getattr(ENGINE, 'has_chat_template', '?')}")
print(f"{'visual token cap applied':<26}: {getattr(ENGINE, 'pixel_cap', '?')}")
print(f"{'MODEL DTYPE':<26}: {getattr(ENGINE, 'quantization', {}).get('param_dtypes')}")
print(f"{'QUANTIZATION CONFIG':<26}: {json.dumps(getattr(ENGINE, 'quantization', {}), default=str)}")
print(f"{'DEVICE MAP':<26}: {getattr(ENGINE, 'device_map_summary', {})}")
print(f"{'parameter devices':<26}: {getattr(ENGINE, 'param_devices', [])}")
print(f"{'first parameter device':<26}: {getattr(ENGINE, 'first_param_device', '?')}")
print(f"{'vision module device':<26}: {getattr(ENGINE, 'vision_device', '?')}")
print(f"{'language model device':<26}: {getattr(ENGINE, 'lm_device', '?')}")
print(f"{'PLACEMENT':<26}: {getattr(ENGINE, 'placement', '?')}")
print(f"{'ACTUAL ATTENTION BACKEND':<26}: {str(getattr(ENGINE, 'attn', '?')).upper()}")
print(f"{'load time':<26}: model {ENGINE.load_seconds}s | processor {ENGINE.processor_seconds}s")
print("\nMEMORY AFTER MODEL LOAD:")
for k, v in snap_after.items():
    if k not in ("tag", "timestamp"):
        print(f"  {k:<26}: {v}")
if getattr(ENGINE, "placement", "") == "MIXED_CPU_GPU":
    print("\n" + "!" * 70)
    print("!! PART OF THE MODEL IS ON CPU. Every forward streams weights over PCIe and a")
    print("!! one-second decode becomes a minute. Lower CFG.RESERVE_VRAM_GIB or free the card.")
    print("!" * 70)

In [ ]:
# =========================================================================
# CELL 16 — CONTROLLED SYNTHETIC OCR TEST  (the gate)
# =========================================================================
# This runs the EXACT pipeline the PDF pages will use, on an image whose content we know.
# If it degenerates, stop here: processing hundreds of pages through a broken multimodal
# interface costs hours and teaches nothing.
OCR_SYSTEM_TEST = ("You read text from images. Transcribe exactly what is written. "
                   "Return only the text.")


def classify_output(text: str, ids: Optional[Sequence[int]] = None) -> Optional[str]:
    """Detect a degenerate generation. Returns a reason, or None if the output looks real."""
    t = (text or "").strip()
    if not t:
        return "EMPTY_OUTPUT"
    if ids is not None and len(ids) > 4 and len(set(ids)) == 1:
        return (f"SINGLE_TOKEN_REPEATED (id={list(ids)[0]}) -- this is argmax over NaN logits: "
                "index 0 is returned every step")
    s = re.sub(r"\s", "", t)
    if s and not re.search(r"[A-Za-z0-9\u0600-\u06FF]", s):
        return f"PUNCTUATION_ONLY ({s[:12]!r})"
    if len(s) > 30 and len(set(s)) <= 2:
        return f"REPEATED_CHARACTER ({s[:8]!r})"
    if len(t) < 3:
        return "EXTREMELY_SHORT"
    return None


def qwen_generate(image: Image.Image, system: str, user: str, max_new_tokens: int,
                  cfg: Config = CFG, engine=None) -> Dict[str, Any]:
    """The single inference path, shared by the sanity test and the page loop."""
    engine = engine or ENGINE
    out: Dict[str, Any] = {"text": "", "ids": [], "input_tokens": None, "visual_tokens": None,
                           "output_tokens": 0, "processor_time_s": 0.0,
                           "generation_time_s": 0.0, "decode_time_s": 0.0,
                           "tokens_per_s": None, "truncated": False, "status": "OK",
                           "error": None}
    if getattr(engine, "is_mock", False):
        out["status"] = "MOCK"
        return out
    messages = [{"role": "system", "content": [{"type": "text", "text": system}]},
                {"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": user}]}]
    try:
        prompt = engine.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=not cfg.DISABLE_THINKING)
    except TypeError:
        prompt = engine.processor.apply_chat_template(messages, tokenize=False,
                                                      add_generation_prompt=True)
    try:
        t0 = time.perf_counter()
        inputs = engine.processor(text=[prompt], images=[image], return_tensors="pt")
        inputs = {k: (v.to(engine.model.device) if hasattr(v, "to") else v)
                  for k, v in inputs.items()}
        out["processor_time_s"] = round(time.perf_counter() - t0, 3)
        out["input_tokens"] = int(inputs["input_ids"].shape[1])
        grid = inputs.get("image_grid_thw")
        if grid is not None:
            try:
                out["visual_tokens"] = int(grid.prod(dim=-1).sum().item() // 4)
            except Exception:
                pass
        if torch.cuda.is_available():
            torch.cuda.synchronize()          # only around the timed GPU region
        t0 = time.perf_counter()
        with torch.inference_mode():
            gen = engine.model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=cfg.DO_SAMPLE, num_beams=1,
                repetition_penalty=cfg.REPETITION_PENALTY,
                no_repeat_ngram_size=cfg.NO_REPEAT_NGRAM_SIZE,
                pad_token_id=getattr(engine.tokenizer, "pad_token_id", None)
                             or getattr(engine.tokenizer, "eos_token_id", None))
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        out["generation_time_s"] = round(time.perf_counter() - t0, 3)
        t0 = time.perf_counter()
        seq = gen[0][inputs["input_ids"].shape[1]:]
        ids = [int(x) for x in seq]
        # RAW: exactly what the model produced. No strip, no normalisation anywhere.
        out["text"] = engine.processor.decode(seq, skip_special_tokens=True,
                                              clean_up_tokenization_spaces=False)
        out["decode_time_s"] = round(time.perf_counter() - t0, 3)
        out["ids"] = ids[:24]
        out["output_tokens"] = len(ids)
        out["truncated"] = len(ids) >= max_new_tokens
        if out["generation_time_s"] > 0:
            out["tokens_per_s"] = round(len(ids) / out["generation_time_s"], 1)
        del gen, inputs
    except Exception as exc:
        out["status"] = "ERROR"
        out["error"] = f"{type(exc).__name__}: {exc}"
        log.error("generation failed: %s", out["error"])
    return out


def make_test_image() -> Image.Image:
    img = Image.new("RGB", (900, 400), "white")
    d = ImageDraw.Draw(img)
    d.rectangle([20, 20, 880, 380], outline="black", width=3)
    d.text((60, 120), "REPUBLIQUE ALGERIENNE", fill="black")
    d.text((60, 180), "NOM: BEN MOHAMED", fill="black")
    d.text((60, 240), "NE LE 01/01/1980  N: 12AB45678", fill="black")
    return img


SANITY: Dict[str, Any] = {"status": "SKIPPED", "reason": "", "diagnosis": []}


def run_sanity_gate(cfg: Config = CFG) -> Dict[str, Any]:
    if getattr(ENGINE, "is_mock", False):
        SANITY.update({"status": "SKIPPED", "reason": "mock engine"})
        print("MOCK engine -- sanity gate skipped")
        return SANITY

    # token id 0 decodes to "!" in Qwen-family byte-level BPE; this confirms the signature exists
    try:
        tok0 = ENGINE.tokenizer.decode([0])
        SANITY["token_id_0"] = repr(tok0)
        print(f"tokenizer.decode([0])      : {tok0!r}"
              f"{'   <- the !!!!!! signature would come from here' if tok0.strip() == '!' else ''}")
    except Exception as exc:
        SANITY["token_id_0"] = f"error: {exc}"

    nan_params = []
    for n, p in ENGINE.model.named_parameters():
        try:
            if torch.isnan(p).any() or torch.isinf(p).any():
                nan_params.append(n)
                if len(nan_params) >= 5:
                    break
        except Exception:
            pass
    SANITY["nan_parameters"] = nan_params
    print(f"NaN/Inf parameters         : {nan_params or 'none'}")

    img = make_test_image()
    user = "Transcribe all text visible in this image."
    gpu_memory_snapshot("before_sanity")
    r = qwen_generate(img, OCR_SYSTEM_TEST, user, cfg.MAX_NEW_TOKENS_TEST
                      if hasattr(cfg, "MAX_NEW_TOKENS_TEST") else 96, cfg)
    snap = gpu_memory_snapshot("after_sanity")

    print("\n" + "=" * 64)
    print("CONTROLLED SYNTHETIC OCR TEST")
    print("=" * 64)
    print(f"INPUT IMAGE        : {img.size[0]}x{img.size[1]}")
    print(f"PROMPT             : {user}")
    print(f"input tokens       : {r['input_tokens']}  (visual: {r['visual_tokens']})")
    print(f"GENERATION TIME    : {r['generation_time_s']}s")
    print(f"OUTPUT TOKENS      : {r['output_tokens']}")
    print(f"TOKENS/SECOND      : {r['tokens_per_s']}")
    print(f"first token ids    : {r['ids'][:16]}")
    print(f"peak GPU allocated : {snap.get('torch_peak_allocated_gib')} GiB")
    print(f"\nRAW MODEL OUTPUT:\n{r['text'][:600]!r}")

    reason = r["error"] if r["status"] == "ERROR" else classify_output(r["text"], r["ids"])
    if reason:
        SANITY.update({"status": "FAILED", "reason": reason, "diagnosis": [
            f"degeneration: {reason}",
            f"1. dtype -- CFG.TORCH_DTYPE={cfg.TORCH_DTYPE}. float16 overflows a VL vision "
            "tower and produces NaN; bfloat16 is correct.",
            f"2. FP8 -- mode={FP8['mode']}, vision excluded via {list(cfg.FP8_SKIP_MODULES)[:3]}"
            "... Set CFG.USE_FP8 = False to isolate the quantiser.",
            f"3. attention -- currently {ENGINE.attn}. Try CFG.ATTENTION_BACKEND = 'eager'.",
            f"4. processor/model pairing -- visual_tokens={r['visual_tokens']}; if this is None "
            "the image placeholders were not expanded and the pairing is wrong.",
            f"5. placement -- {getattr(ENGINE, 'placement', '?')}.",
        ]})
        print("\n" + "=" * 64)
        print("SANITY TEST FAILED -- do NOT process documents yet")
        print("=" * 64)
        for line in SANITY["diagnosis"]:
            print(" -", line)
    else:
        SANITY.update({"status": "PASSED", "reason": "output is real text",
                       "tokens_per_s": r["tokens_per_s"]})
        print("\nSANITY TEST PASSED -- the model sees the image and returns real text")
    return SANITY


CFG.MAX_NEW_TOKENS_TEST = 96
SANITY = run_sanity_gate(CFG)
print(f"\nSANITY STATUS = {SANITY['status']}")

In [ ]:
# =========================================================================
# CELL 17 / 18 — RANDOM CUSTOMER SELECTION AND PDF INVENTORY
# =========================================================================
def select_customer(cfg: Config = CFG) -> Path:
    root = Path(cfg.CUSTOMER_ROOT)
    if not root.exists():
        raise FileNotFoundError(f"CUSTOMER_ROOT does not exist: {root}")
    folders = sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.name)
    if not folders:
        raise FileNotFoundError(f"no customer folders under {root}")
    if cfg.CUSTOMER_ID:
        for f in folders:
            if f.name == cfg.CUSTOMER_ID:
                return f
        raise ValueError(f"CUSTOMER_ID {cfg.CUSTOMER_ID!r} not found among {len(folders)} folders")
    chosen = random.Random(cfg.RANDOM_SEED).choice(folders)   # deterministic for a given seed
    log.info("random customer (seed %s of %d folders): %s", cfg.RANDOM_SEED, len(folders),
             chosen.name)
    return chosen


def build_inventory(customer_dir: Path, cfg: Config = CFG) -> pd.DataFrame:
    """EVERY PDF in the folder, with its ACTUAL page count. No filename assumptions, no
    duplicate filtering, no classification -- we only record what exists."""
    rows = []
    pdfs = sorted([p for p in customer_dir.rglob("*") if p.suffix.lower() == ".pdf"],
                  key=lambda p: p.name)
    for p in pdfs:
        row = {"customer_id": customer_dir.name, "filename": p.name, "path": str(p),
               "file_size_bytes": p.stat().st_size,
               "file_size_mb": round(p.stat().st_size / 2**20, 3),
               "page_count": None, "open_status": "OK", "render_status": "PENDING",
               "error": None}
        t0 = time.perf_counter()
        try:
            doc = pymupdf.open(str(p))
            row["page_count"] = len(doc)          # discovered, never assumed
            doc.close()
        except Exception as exc:
            row.update({"open_status": "FAILED", "error": f"{type(exc).__name__}: {exc}"})
        row["open_time_s"] = round(time.perf_counter() - t0, 4)
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(DIRS["reports"] / "document_inventory.csv", index=False, encoding="utf-8-sig")
    return df


print("17/18 ready: select_customer(), build_inventory()")

In [ ]:
# =========================================================================
# CELL 19 / 20 — PAGE RENDERING AND IMAGE QUALITY ANALYSIS
# =========================================================================
def render_page(doc, index: int, cfg: Config = CFG) -> Dict[str, Any]:
    """Render ONE page once. The returned image is the ORIGINAL and is never modified."""
    t0 = time.perf_counter()
    page = doc[index]
    rect = page.rect
    pix = page.get_pixmap(dpi=cfg.RENDER_DPI, colorspace=pymupdf.csRGB, alpha=False)
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
    return {"original": img, "render_time_s": round(time.perf_counter() - t0, 3),
            "render_width": pix.width, "render_height": pix.height,
            "pdf_width_pt": round(rect.width, 1), "pdf_height_pt": round(rect.height, 1),
            "render_dpi": cfg.RENDER_DPI}


def analyze_quality(img: Image.Image) -> Dict[str, Any]:
    """Cheap indicators that DRIVE preprocessing decisions. They never touch the text content.

    Measured on a fixed 1000 px working copy: variance of Laplacian scales with image size, so
    thresholds are meaningless unless the measurement scale is fixed."""
    g = img.convert("L")
    if max(g.size) > 1000:
        s = 1000 / max(g.size)
        g = g.resize((max(1, int(g.width * s)), max(1, int(g.height * s))), Image.BILINEAR)
    a = np.asarray(g, dtype=np.uint8)
    p5, p50, p95 = np.percentile(a, [5, 50, 95])
    q = {"width": img.width, "height": img.height,
         "aspect_ratio": round(img.width / max(1, img.height), 3),
         "gray_mean": round(float(a.mean()), 1),
         "gray_median": round(float(p50), 1),
         "contrast_p95_p5": round(float((p95 - p5) / 255.0), 3),
         "std": round(float(a.std()), 1),
         "dark_pixel_pct": round(float((a < 40).mean() * 100), 2),
         "bright_pixel_pct": round(float((a > 215).mean() * 100), 2)}
    if cv2 is not None:
        q["blur_laplacian_var"] = round(float(cv2.Laplacian(a, cv2.CV_64F).var()), 1)
        # Immerkaer noise estimate: one 3x3 convolution, separates sensor noise from structure
        M = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
        conv = cv2.filter2D(a.astype(np.float32), -1, M)
        h, w = a.shape
        q["noise_sigma"] = round(float(np.abs(conv).sum() * math.sqrt(0.5 * math.pi) /
                                       (6.0 * (w - 2) * (h - 2))), 2)
        binv = cv2.threshold(a, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        q["ink_pct"] = round(float((binv > 0).mean() * 100), 2)
        # median height of text-like components: predicts OCR failure better than DPI does
        n, _, stats, _ = cv2.connectedComponentsWithStats(binv, connectivity=8)
        H = a.shape[0]
        hs = [stats[i][3] for i in range(1, n)
              if 3 <= stats[i][3] <= max(8, H * 0.08) and stats[i][4] >= 6]
        scale_back = max(img.size) / max(a.shape)
        q["text_height_px"] = round(float(np.median(hs)) * scale_back, 1) if len(hs) >= 8 else 0.0
        # illumination uniformity: high value = shadows / uneven lighting
        bh, bw = max(1, a.shape[0] // 12), max(1, a.shape[1] // 12)
        blocks = [a[i:i + bh, j:j + bw].mean() for i in range(0, a.shape[0] - bh + 1, bh)
                  for j in range(0, a.shape[1] - bw + 1, bw)]
        q["illumination_std"] = round(float(np.std(blocks) / 255.0), 4) if blocks else 0.0
    else:
        q.update({"blur_laplacian_var": None, "noise_sigma": None, "ink_pct": None,
                  "text_height_px": 0.0, "illumination_std": None})

    # ---- recommendations, not actions ----
    q["low_contrast"] = bool(q["contrast_p95_p5"] < 0.35)
    q["uneven_illumination"] = bool((q["illumination_std"] or 0) > 0.13)
    q["noisy"] = bool((q["noise_sigma"] or 0) > 6.0)
    q["blurred"] = bool((q["blur_laplacian_var"] or 999) < 90)
    q["dark_scan"] = bool(q["gray_mean"] < 100)
    q["faded"] = bool(q["gray_mean"] > 205 and q["contrast_p95_p5"] < 0.45)
    q["small_text"] = bool(0 < q["text_height_px"] < 20)
    q["grade"] = ("poor" if sum([q["low_contrast"], q["noisy"], q["blurred"]]) >= 2 else
                  "fair" if any([q["low_contrast"], q["noisy"], q["blurred"]]) else "good")
    return q


print("19/20 ready: render_page(), analyze_quality()")

In [ ]:
# =========================================================================
# CELL 21 — IMAGE PREPROCESSING  (conservative, configurable, original preserved)
# =========================================================================
# Every step is optional and most are gated on a MEASUREMENT, not applied blindly. The order
# matters: geometry first (rotation, deskew, border), then resize, then photometry on the
# smaller image, then upscale only if glyphs are genuinely too small.
#
# USE_THRESHOLD is OFF by default. Binarisation is the single most destructive operation for
# these documents: it eats thin strokes, MRZ characters, French accents and pen lines.
def _to_gray_np(img: Image.Image, side: int = 1200) -> np.ndarray:
    g = img.convert("L")
    if max(g.size) > side:
        s = side / max(g.size)
        g = g.resize((max(1, int(g.width * s)), max(1, int(g.height * s))), Image.BILINEAR)
    return np.asarray(g, dtype=np.uint8)


def detect_rotation(img: Image.Image, cfg: Config = CFG) -> Dict[str, Any]:
    """0/90/180/270 from text-line geometry. Abstains when the margin is small: rotating a page
    on a coin flip is worse than leaving it alone."""
    out = {"rotation": 0, "margin": 0.0, "method": "disabled"}
    if not cfg.USE_AUTO_ROTATION or cv2 is None:
        return out
    a = _to_gray_np(img, 900)
    binv = cv2.threshold(a, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    def axis_ratio(m):
        row, col = m.sum(axis=1).astype(np.float32), m.sum(axis=0).astype(np.float32)
        rv = row.var() / (row.mean() ** 2 + 1e-6)
        cv_ = col.var() / (col.mean() ** 2 + 1e-6)
        return (rv + 1e-6) / (cv_ + 1e-6)

    def updown(m):
        """Ink sits above the baseline in Latin, Cyrillic and Arabic, so a text band's centroid
        is above its geometric centre when the page is upright."""
        row = m.sum(axis=1).astype(np.float32)
        if row.max() <= 0:
            return 0.0
        thr = row.mean() + 0.3 * row.std()
        bands, start = [], None
        for i, v in enumerate(row):
            if v > thr and start is None:
                start = i
            elif v <= thr and start is not None:
                if i - start >= 3:
                    bands.append((start, i))
                start = None
        if len(bands) < 3:
            return 0.0
        sc = []
        for s0, s1 in bands[:60]:
            seg = row[s0:s1]
            if seg.sum() <= 0:
                continue
            idx = np.arange(len(seg), dtype=np.float32)
            sc.append(0.5 - float((seg * idx).sum() / seg.sum()) / max(1, len(seg) - 1))
        return float(np.mean(sc) * 2) if sc else 0.0

    horizontal = axis_ratio(binv) >= 1.0
    cands = (0, 180) if horizontal else (90, 270)
    scores = {d: updown(np.ascontiguousarray(np.rot90(binv, k=d // 90) if d else binv))
              for d in cands}
    best = max(scores, key=lambda d: scores[d])
    margin = abs(scores[cands[0]] - scores[cands[1]])
    out.update({"method": "projection+baseline", "margin": round(float(margin), 4),
                "axis_horizontal": bool(horizontal), "scores": {k: round(v, 4)
                                                                for k, v in scores.items()}})
    # abstain when uncertain -- but a 90/270 axis finding is geometric and trustworthy
    if best in (90, 270) or margin >= cfg.ROTATION_MIN_MARGIN:
        out["rotation"] = int(best)
    else:
        out["abstained"] = True
    return out


def detect_skew(img: Image.Image, cfg: Config = CFG) -> float:
    if not cfg.USE_DESKEW or cv2 is None:
        return 0.0
    a = _to_gray_np(img, 1200)
    binv = cv2.threshold(a, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    binv = cv2.dilate(binv, cv2.getStructuringElement(cv2.MORPH_RECT, (25, 3)))
    coords = cv2.findNonZero(binv)
    if coords is None or len(coords) < 80:
        return 0.0
    ang = cv2.minAreaRect(coords)[-1]
    ang = ang + 90 if ang < -45 else (ang - 90 if ang > 45 else ang)
    return 0.0 if abs(ang) > cfg.DESKEW_MAX_DEG else round(float(ang), 2)


def crop_border(img: Image.Image) -> Optional[Tuple[int, int, int, int]]:
    """Trim uniform scanner borders (black edges, wide margins). Conservative: refuses to cut
    more than 15% of a side, so no content is lost."""
    if cv2 is None:
        return None
    a = _to_gray_np(img, 1000)
    binv = cv2.threshold(a, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    cnt = cv2.findNonZero(binv)
    if cnt is None:
        return None
    x, y, w, h = cv2.boundingRect(cnt)
    H, W = a.shape
    if w < 0.5 * W or h < 0.5 * H:
        return None
    pad = 0.01
    x0, y0 = max(0, x - int(pad * W)), max(0, y - int(pad * H))
    x1, y1 = min(W, x + w + int(pad * W)), min(H, y + h + int(pad * H))
    if x0 > 0.15 * W or y0 > 0.15 * H or x1 < 0.85 * W or y1 < 0.85 * H:
        return None
    sx, sy = img.width / W, img.height / H
    return (int(x0 * sx), int(y0 * sy), int(x1 * sx), int(y1 * sy))


def preprocess_page(original: Image.Image, quality: Dict[str, Any],
                    cfg: Config = CFG) -> Tuple[Image.Image, Dict[str, Any]]:
    """Returns (preprocessed_image, metadata). `original` is NEVER modified."""
    t0 = time.perf_counter()
    img = original.copy()                      # the original stays untouched
    applied: List[str] = []
    meta: Dict[str, Any] = {"input_size": list(original.size), "rotation": 0, "skew_deg": 0.0}

    rot = detect_rotation(img, cfg)
    meta["rotation_detection"] = rot
    if rot["rotation"]:
        img = img.rotate(rot["rotation"], expand=True)     # PIL rotates counter-clockwise
        meta["rotation"] = rot["rotation"]
        applied.append(f"rotate_{rot['rotation']}")

    skew = detect_skew(img, cfg)
    if abs(skew) >= 0.3 and cv2 is not None:
        a = np.asarray(img)
        h, w = a.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), skew, 1.0)
        cos, sin = abs(M[0, 0]), abs(M[0, 1])
        nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
        M[0, 2] += nw / 2 - w / 2
        M[1, 2] += nh / 2 - h / 2
        img = Image.fromarray(cv2.warpAffine(a, M, (nw, nh), flags=cv2.INTER_CUBIC,
                                             borderMode=cv2.BORDER_REPLICATE))
        meta["skew_deg"] = skew
        applied.append(f"deskew_{skew:+.2f}")

    if cfg.USE_BORDER_CLEANUP:
        box = crop_border(img)
        if box:
            img = img.crop(box)
            meta["border_crop"] = list(box)
            applied.append("border_cleanup")

    # resize BEFORE photometry: filtering pixels that are about to be discarded is wasted time
    if max(img.size) > cfg.MAX_IMAGE_DIMENSION:
        s = cfg.MAX_IMAGE_DIMENSION / max(img.size)
        img = img.resize((int(img.width * s), int(img.height * s)), Image.LANCZOS)
        applied.append(f"resize_{cfg.MAX_IMAGE_DIMENSION}")

    if cv2 is not None:
        g = np.asarray(img.convert("L"))
        if cfg.USE_GRAYSCALE:
            applied.append("grayscale")
        if cfg.USE_BRIGHTNESS and quality.get("uneven_illumination"):
            bg = cv2.GaussianBlur(g.astype(np.float32), (0, 0), sigmaX=max(g.shape) / 30.0)
            g = np.clip(g / (bg + 1e-3) * float(np.median(bg)), 0, 255).astype(np.uint8)
            applied.append("flatten_illumination")
        if cfg.USE_CONTRAST and (quality.get("low_contrast") or quality.get("faded")
                                 or quality.get("dark_scan")):
            g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
            applied.append("clahe")
        if cfg.USE_DENOISE and quality.get("noisy"):
            g = cv2.bilateralFilter(g, 5, 45, 45)     # edge preserving; NOT median/Gaussian
            applied.append("bilateral_denoise")
        if cfg.USE_SHARPEN:
            blur = cv2.GaussianBlur(g, (0, 0), 1.2)
            g = cv2.addWeighted(g, 1.4, blur, -0.4, 0)   # mild unsharp: linear, invents nothing
            applied.append("unsharp_mild")
        if cfg.USE_THRESHOLD:
            g = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY, 31, 11)
            applied.append("adaptive_threshold")
        img = Image.fromarray(g).convert("RGB")

    # upscale only if the glyphs the model will see are genuinely too small
    factor = cfg.UPSCALE_FACTOR
    th = quality.get("text_height_px") or 0
    if factor <= 1.0 and 0 < th < cfg.MIN_TEXT_HEIGHT_PX:
        factor = min(2.0, cfg.MIN_TEXT_HEIGHT_PX / th)
    if factor > 1.02:
        img = img.resize((int(img.width * factor), int(img.height * factor)), Image.LANCZOS)
        applied.append(f"upscale_x{factor:.2f}")

    meta.update({"applied": applied, "output_size": list(img.size),
                 "approx_visual_tokens": int(img.width * img.height / 784),
                 "preprocess_time_s": round(time.perf_counter() - t0, 3)})
    return img, meta


print("21 ready: detect_rotation(), detect_skew(), preprocess_page()")

In [ ]:
# =========================================================================
# CELL 22 — OCR PROMPT (Algerian administrative / KYC documents)
# =========================================================================
# Contextual knowledge helps the model recognise what a label IS. It must never let the model
# supply what the pixels do not show -- hence the explicit prohibitions.
OCR_SYSTEM = """You are reading an image of a scanned Algerian administrative or KYC document.

Transcribe the visible text exactly as it appears. The image is the ONLY source of truth.

These documents are often photocopies: black and white, faded, skewed, compressed, stamped, and mixing printed and handwritten text. They may be Algerian identity cards, passports, residence certificates, bank forms or FATCA forms, in French, in Arabic, or both. You may see labels such as Nom, Prenom, Ne(e) le, Lieu de naissance, Commune, Daira, Wilaya, Nationalite, Sexe, Delivre le, Expire le, Numero de compte, and names written in Latin or Arabic script.

PRESERVE
- French exactly as written, including accents
- Arabic exactly as written, in Arabic script
- Latin names exactly as written, including every component
- numbers, dates and punctuation exactly as written
- stamped text when readable
- handwritten text when readable
- MRZ characters exactly, including every <

DO NOT
- guess or complete missing characters
- correct spelling
- translate
- transliterate between Arabic and Latin
- normalize names or reorder them
- invent unreadable text
- use outside knowledge of Algerian documents to fill anything in
- infer text that is not visible

If a portion is genuinely unreadable, write [UNREADABLE] instead of guessing.

Preserve the approximate reading order. Return ONLY the transcription: no explanation, no description of the document, no JSON."""

OCR_USER = "Transcribe all text visible in this document page."

print(f"system prompt: {len(OCR_SYSTEM)} chars")
print(f"user prompt  : {OCR_USER!r}")
print("output format: plain text (no JSON, no schema, no fields)")

In [ ]:
# =========================================================================
# CELL 23 / 24 — PAGE OCR WITH FULL PER-STAGE TIMING
# =========================================================================
RESULTS: List[Dict[str, Any]] = []
ERRORS: List[Dict[str, Any]] = []
QWEN_CALLS = {"n": 0}
RUN_STATE = {"deadline": None, "start": None}


def safe_stem(s: str) -> str:
    """Filename-safe stem. Affects FILENAMES only -- never the transcription."""
    return re.sub(r"[^A-Za-z0-9._ -]+", "_", Path(s).stem).strip() or "document"


def ocr_page(customer_id: str, pdf_path: Path, doc, index: int, page_count: int,
             cfg: Config = CFG) -> Dict[str, Any]:
    """Render -> analyse -> preprocess -> ONE Qwen call. Every stage timed separately."""
    t_page = time.perf_counter()
    page_no = index + 1
    rec: Dict[str, Any] = {"customer_id": customer_id, "pdf": pdf_path.name,
                           "page": page_no, "page_count": page_count}

    r = render_page(doc, index, cfg)
    original = r["original"]
    rec.update({k: v for k, v in r.items() if k != "original"})

    t0 = time.perf_counter()
    quality = analyze_quality(original)
    rec["quality_time_s"] = round(time.perf_counter() - t0, 3)
    rec["quality"] = quality

    processed, pmeta = preprocess_page(original, quality, cfg)
    rec["preprocess_time_s"] = pmeta["preprocess_time_s"]
    rec["preprocessing"] = pmeta
    rec["sent_width"], rec["sent_height"] = processed.size

    debug_paths = {}
    t0 = time.perf_counter()
    if cfg.SAVE_DEBUG_IMAGES:
        base = f"{customer_id}__{safe_stem(pdf_path.name)}__p{page_no:03d}"
        o = DIRS["debug"] / f"{base}__original.png"
        p = DIRS["debug"] / f"{base}__preprocessed.png"
        og = original.copy()
        og.thumbnail((1600, 1600))
        og.save(o)
        processed.save(p)
        (DIRS["debug"] / f"{base}__preprocessing.json").write_text(
            json.dumps({"quality": quality, "preprocessing": pmeta}, indent=2, default=str),
            encoding="utf-8")
        debug_paths = {"original_png": str(o), "preprocessed_png": str(p)}
    rec["debug_write_time_s"] = round(time.perf_counter() - t0, 3)
    rec["debug"] = debug_paths

    QWEN_CALLS["n"] += 1
    call_no = QWEN_CALLS["n"]
    g = qwen_generate(processed, OCR_SYSTEM, OCR_USER, cfg.MAX_NEW_TOKENS, cfg)
    status = classify_output(g["text"], g["ids"]) if g["status"] == "OK" else g["error"]
    rec.update({
        "qwen_call": call_no, "processor_time_s": g["processor_time_s"],
        "generation_time_s": g["generation_time_s"], "decode_time_s": g["decode_time_s"],
        "input_tokens": g["input_tokens"], "visual_tokens": g["visual_tokens"],
        "output_tokens": g["output_tokens"], "tokens_per_s": g["tokens_per_s"],
        "truncated": g["truncated"], "max_new_tokens": cfg.MAX_NEW_TOKENS,
        "raw_model_output": g["text"],                       # verbatim, never normalised
        "ocr_status": "DEGRADED" if status else "OUTPUT_RECEIVED",
        "degradation_reason": status, "error": g["error"]})

    t0 = time.perf_counter()
    txt = DIRS["raw"] / f"{customer_id}__{safe_stem(pdf_path.name)}__p{page_no:03d}.txt"
    txt.write_text(g["text"], encoding="utf-8")
    rec["io_time_s"] = round(time.perf_counter() - t0, 3)
    rec["txt_path"] = str(txt)
    rec["gpu"] = gpu_memory_snapshot(f"page_{call_no}")
    rec["total_time_s"] = round(time.perf_counter() - t_page, 3)

    print(f"[PAGE {call_no}] {customer_id} | {pdf_path.name} | page {page_no}/{page_count} | "
          f"orig {r['render_width']}x{r['render_height']} -> sent {processed.size[0]}x"
          f"{processed.size[1]} | prep={';'.join(pmeta['applied']) or 'none'} | "
          f"vis_tok={g['visual_tokens']} out={g['output_tokens']}tok "
          f"gen={g['generation_time_s']}s total={rec['total_time_s']}s | {rec['ocr_status']}",
          flush=True)
    if rec["total_time_s"] > cfg.WARN_CALL_SECONDS:
        print(f"  SLOW: placement={getattr(ENGINE, 'placement', '?')} "
              f"visual_tokens={g['visual_tokens']} tokens/s={g['tokens_per_s']}", flush=True)
    if status:
        print(f"  DEGRADED OUTPUT: {status}", flush=True)
        ERRORS.append({"timestamp": utcnow(), "customer_id": customer_id, "pdf": pdf_path.name,
                       "page": page_no, "stage": "qwen", "error": status})
    return rec


print("23/24 ready: ocr_page()")

In [ ]:
# =========================================================================
# CELL 25 / 26 — DRY RUN AND REAL EXECUTION
# =========================================================================
CUSTOMER_DIR = select_customer(CFG)
INVENTORY = build_inventory(CUSTOMER_DIR, CFG)
TOTAL_PAGES = int(INVENTORY["page_count"].fillna(0).sum())

print("=" * 64)
print("QWEN3.6 RAW OCR EXPERIMENT -- PLAN")
print("=" * 64)
print(f"SELECTED CUSTOMER : {CUSTOMER_DIR.name}")
print(f"PDFs              : {len(INVENTORY)}")
print()
print(INVENTORY[["filename", "file_size_mb", "page_count", "open_status"]].to_string(index=False))
print(f"\nTOTAL PAGES       : {TOTAL_PAGES}")
print(f"EXPECTED QWEN CALLS: {TOTAL_PAGES}   (one call per page)")
print(f"limits            : MAX_QWEN_CALLS={CFG.MAX_QWEN_CALLS}, "
      f"MAX_RUNTIME_SECONDS={CFG.MAX_RUNTIME_SECONDS}")
print(f"output directory  : {CFG.OUTPUT_DIR}")
print("\npreprocessing configuration:")
for k in ("USE_AUTO_ROTATION", "USE_DESKEW", "USE_GRAYSCALE", "USE_CONTRAST", "USE_BRIGHTNESS",
          "USE_DENOISE", "USE_SHARPEN", "USE_THRESHOLD", "USE_BORDER_CLEANUP",
          "USE_PERSPECTIVE", "UPSCALE_FACTOR"):
    print(f"  {k:<22}: {getattr(CFG, k)}")
print(f"  {'RENDER_DPI':<22}: {CFG.RENDER_DPI}")
print(f"  {'MAX_IMAGE_DIMENSION':<22}: {CFG.MAX_IMAGE_DIMENSION}")
print(f"  {'MAX_NEW_TOKENS':<22}: {CFG.MAX_NEW_TOKENS}")
print(f"  {'SAVE_DEBUG_IMAGES':<22}: {CFG.SAVE_DEBUG_IMAGES}")
print("=" * 64)

if TOTAL_PAGES > CFG.MAX_QWEN_CALLS:
    print(f"\nWARNING: {TOTAL_PAGES} pages exceeds MAX_QWEN_CALLS={CFG.MAX_QWEN_CALLS}; "
          "the run will stop gracefully at the limit.")

RUN_SECONDS = [0.0]
WARMUP: Dict[str, Any] = {"ran": False}

if CFG.DRY_RUN:
    print("\nDRY_RUN is ON -- no Qwen call was made.")
    print("Review the plan above, then set CFG.DRY_RUN = False and re-run this cell.")
elif SANITY["status"] == "FAILED":
    print("\nSANITY TEST FAILED -- refusing to process documents. See CELL 16 diagnosis.")
else:
    # ---- warmup, measured separately and never counted as a page ----
    if CFG.RUN_WARMUP and not getattr(ENGINE, "is_mock", False):
        gpu_memory_snapshot("before_warmup")
        t0 = time.perf_counter()
        w = qwen_generate(Image.new("RGB", (768, 512), "white"), OCR_SYSTEM_TEST,
                          "Describe this image in one word.", 32, CFG)
        WARMUP = {"ran": True, "warmup_time_s": round(time.perf_counter() - t0, 2),
                  "warmup_tokens": w["output_tokens"], "warmup_tokens_per_s": w["tokens_per_s"],
                  "gpu": gpu_memory_snapshot("after_warmup")}
        print(f"\nWARMUP TIME       : {WARMUP['warmup_time_s']}s")
        print(f"WARMUP TOKENS     : {WARMUP['warmup_tokens']}")
        print(f"WARMUP TOKENS/SEC : {WARMUP['warmup_tokens_per_s']}")
        print(f"GPU after warmup  : allocated "
              f"{WARMUP['gpu'].get('torch_allocated_gib')} GiB, peak "
              f"{WARMUP['gpu'].get('torch_peak_allocated_gib')} GiB")

    RUN_STATE["start"] = time.perf_counter()
    RUN_STATE["deadline"] = RUN_STATE["start"] + CFG.MAX_RUNTIME_SECONDS
    stopped = None
    print("\nstarting OCR\n")
    for _, row in INVENTORY.iterrows():
        if row["open_status"] != "OK":
            continue
        pdf_path = Path(row["path"])
        try:
            doc = pymupdf.open(str(pdf_path))
        except Exception as exc:
            ERRORS.append({"timestamp": utcnow(), "customer_id": CUSTOMER_DIR.name,
                           "pdf": pdf_path.name, "page": None, "stage": "open",
                           "error": f"{type(exc).__name__}: {exc}"})
            continue
        try:
            n_pages = min(len(doc), CFG.MAX_PAGES_PER_PDF)
            for i in range(n_pages):
                if QWEN_CALLS["n"] >= CFG.MAX_QWEN_CALLS:
                    stopped = "MAX_QWEN_CALLS"
                    break
                if time.perf_counter() > RUN_STATE["deadline"]:
                    stopped = "MAX_RUNTIME_SECONDS"
                    break
                try:
                    RESULTS.append(ocr_page(CUSTOMER_DIR.name, pdf_path, doc, i, len(doc), CFG))
                except Exception as exc:
                    ERRORS.append({"timestamp": utcnow(), "customer_id": CUSTOMER_DIR.name,
                                   "pdf": pdf_path.name, "page": i + 1, "stage": "ocr_page",
                                   "error": f"{type(exc).__name__}: {exc}",
                                   "traceback": traceback.format_exc()})
                    log.error("page %d of %s failed: %s", i + 1, pdf_path.name, exc)
        finally:
            doc.close()
        if stopped:
            break
    RUN_SECONDS[0] = time.perf_counter() - RUN_STATE["start"]
    if stopped:
        last = RESULTS[-1] if RESULTS else {}
        print(f"\nSTOPPED GRACEFULLY: {stopped}")
        print(f"  customer={CUSTOMER_DIR.name} pdf={last.get('pdf')} page={last.get('page')}")
        print(f"  Qwen calls completed={QWEN_CALLS['n']} elapsed={RUN_SECONDS[0]:.1f}s")
    gpu_memory_snapshot("after_ocr")
    print(f"\nOCR finished: {len(RESULTS)} pages in {RUN_SECONDS[0]:.1f}s")

In [ ]:
# =========================================================================
# CELL 27 / 28 — OUTPUT FILES AND PERFORMANCE REPORTS
# =========================================================================
def write_outputs(cfg: Config = CFG) -> Dict[str, Path]:
    rep, paths = DIRS["reports"], {}

    # human-readable transcript
    combined = DIRS["raw"] / "raw_ocr_results.txt"
    with combined.open("w", encoding="utf-8") as fh:
        for r in RESULTS:
            fh.write("=" * 60 + "\n")
            fh.write(f"CUSTOMER: {r['customer_id']}\n")
            fh.write(f"PDF: {r['pdf']}\n")
            fh.write(f"PAGE: {r['page']}/{r['page_count']}\n")
            fh.write("=" * 60 + "\n\n")
            fh.write(r["raw_model_output"])      # verbatim
            fh.write("\n\n")
    paths["txt"] = combined

    with (DIRS["raw"] / "raw_ocr_results.jsonl").open("w", encoding="utf-8") as fh:
        for r in RESULTS:
            fh.write(json.dumps({
                "customer_id": r["customer_id"], "pdf": r["pdf"], "page": r["page"],
                "page_count": r["page_count"],
                "image_width": r.get("sent_width"), "image_height": r.get("sent_height"),
                "render_width": r.get("render_width"), "render_height": r.get("render_height"),
                "preprocessing_configuration": r.get("preprocessing", {}).get("applied"),
                "preprocessing_detail": r.get("preprocessing"),
                "quality": r.get("quality"),
                "raw_model_output": r["raw_model_output"],
                "generation_time": r.get("generation_time_s"),
                "output_tokens": r.get("output_tokens"),
                "tokens_per_second": r.get("tokens_per_s"),
                "gpu_memory": r.get("gpu"),
                "ocr_status": r.get("ocr_status"),
                "error": r.get("error")}, ensure_ascii=False, default=str) + "\n")
    paths["jsonl"] = DIRS["raw"] / "raw_ocr_results.jsonl"

    if RESULTS:
        df = pd.DataFrame([{
            "customer_id": r["customer_id"], "pdf": r["pdf"], "page": r["page"],
            "page_count": r["page_count"], "raw_model_output": r["raw_model_output"],
            "ocr_status": r["ocr_status"], "degradation_reason": r.get("degradation_reason"),
            "output_tokens": r.get("output_tokens"), "generation_time_s": r.get("generation_time_s"),
            "total_time_s": r.get("total_time_s")} for r in RESULTS])
        df.to_csv(rep / "raw_ocr_results.csv", index=False, encoding="utf-8-sig")
        perf = pd.DataFrame([{
            "customer_id": r["customer_id"], "pdf": r["pdf"], "page": r["page"],
            "render_time_s": r.get("render_time_s"), "quality_time_s": r.get("quality_time_s"),
            "preprocess_time_s": r.get("preprocess_time_s"),
            "processor_time_s": r.get("processor_time_s"),
            "generation_time_s": r.get("generation_time_s"),
            "decode_time_s": r.get("decode_time_s"),
            "debug_write_time_s": r.get("debug_write_time_s"), "io_time_s": r.get("io_time_s"),
            "total_time_s": r.get("total_time_s"), "input_tokens": r.get("input_tokens"),
            "visual_tokens": r.get("visual_tokens"), "output_tokens": r.get("output_tokens"),
            "tokens_per_s": r.get("tokens_per_s"), "truncated": r.get("truncated"),
            "render_width": r.get("render_width"), "render_height": r.get("render_height"),
            "sent_width": r.get("sent_width"), "sent_height": r.get("sent_height"),
            "rotation": r.get("preprocessing", {}).get("rotation"),
            "skew_deg": r.get("preprocessing", {}).get("skew_deg"),
            "preprocessing": ";".join(r.get("preprocessing", {}).get("applied", [])),
            "quality_grade": r.get("quality", {}).get("grade"),
            "text_height_px": r.get("quality", {}).get("text_height_px"),
            "contrast": r.get("quality", {}).get("contrast_p95_p5"),
            "blur_var": r.get("quality", {}).get("blur_laplacian_var"),
            "ocr_status": r.get("ocr_status")} for r in RESULTS])
        perf.to_csv(rep / "performance_by_page.csv", index=False, encoding="utf-8-sig")
        paths["results_csv"] = rep / "raw_ocr_results.csv"
        paths["performance_by_page"] = rep / "performance_by_page.csv"

        def q(col, f):
            s = pd.to_numeric(perf[col], errors="coerce").dropna()
            return round(float(f(s)), 3) if len(s) else None

        gen = pd.to_numeric(perf["generation_time_s"], errors="coerce").dropna()
        summary = {
            "customer_id": CUSTOMER_DIR.name, "pdfs": int(len(INVENTORY)),
            "pages_available": TOTAL_PAGES, "pages_processed": len(RESULTS),
            "qwen_calls": QWEN_CALLS["n"],
            "successful_calls": int((perf["ocr_status"] == "OUTPUT_RECEIVED").sum()),
            "degraded_calls": int((perf["ocr_status"] == "DEGRADED").sum()),
            "total_runtime_s": round(RUN_SECONDS[0], 1),
            "total_qwen_time_s": round(float(gen.sum()), 1) if len(gen) else 0.0,
            "avg_qwen_latency_s": round(float(gen.mean()), 2) if len(gen) else None,
            "median_qwen_latency_s": round(float(gen.median()), 2) if len(gen) else None,
            "p95_qwen_latency_s": round(float(gen.quantile(0.95)), 2) if len(gen) else None,
            "total_render_time_s": q("render_time_s", lambda s: s.sum()),
            "total_preprocess_time_s": q("preprocess_time_s", lambda s: s.sum()),
            "total_processor_time_s": q("processor_time_s", lambda s: s.sum()),
            "total_io_time_s": q("io_time_s", lambda s: s.sum()),
            "avg_output_tokens": q("output_tokens", lambda s: s.mean()),
            "avg_tokens_per_s": q("tokens_per_s", lambda s: s.mean()),
            "avg_visual_tokens": q("visual_tokens", lambda s: s.mean()),
            "pages_truncated": int(perf["truncated"].fillna(False).sum()),
            "gpu_peak_allocated_gib": max((m.get("torch_peak_allocated_gib") or 0
                                           for m in MEMORY_TRACE), default=None),
            "model": getattr(ENGINE, "name", "?"),
            "attention_backend": getattr(ENGINE, "attn", "?"),
            "placement": getattr(ENGINE, "placement", "?"),
            "render_dpi": cfg.RENDER_DPI, "max_new_tokens": cfg.MAX_NEW_TOKENS,
            "warmup_time_s": WARMUP.get("warmup_time_s")}
        pd.DataFrame([summary]).to_csv(rep / "performance_summary.csv", index=False,
                                       encoding="utf-8-sig")
        paths["performance_summary"] = rep / "performance_summary.csv"
    else:
        summary = {}

    pd.DataFrame(ERRORS, columns=["timestamp", "customer_id", "pdf", "page", "stage", "error",
                                  "traceback"]).to_csv(rep / "errors.csv", index=False,
                                                       encoding="utf-8-sig")
    paths["errors"] = rep / "errors.csv"
    paths["inventory"] = rep / "document_inventory.csv"
    pd.DataFrame(MEMORY_TRACE).to_csv(rep / "gpu_memory_trace.csv", index=False,
                                      encoding="utf-8-sig")
    paths["gpu_memory_trace"] = rep / "gpu_memory_trace.csv"
    return paths, summary


if not CFG.DRY_RUN and RESULTS:
    PATHS, SUMMARY = write_outputs(CFG)
    print("outputs:")
    for k, v in PATHS.items():
        print(f"  {k:<22}: {v}")
else:
    PATHS, SUMMARY = {}, {}
    print("nothing to write (DRY_RUN or no results)")

In [ ]:
# =========================================================================
# CELL 29 / 30 — DEGRADATION DIAGNOSTICS AND FINAL SUMMARY
# =========================================================================
if SUMMARY:
    s = SUMMARY
    print("=" * 60)
    print("QWEN3.6 RAW OCR EXPERIMENT")
    print("=" * 60)
    print(f"Customer:              {s['customer_id']}")
    print(f"PDFs:                  {s['pdfs']}")
    print(f"Pages:                 {s['pages_processed']} of {s['pages_available']}")
    print(f"Successful OCR calls:  {s['successful_calls']}")
    print(f"Failed/degraded calls: {s['degraded_calls']}")
    print()
    print(f"Total runtime:         {s['total_runtime_s']} s")
    print(f"Total Qwen time:       {s['total_qwen_time_s']} s")
    print()
    print(f"Average Qwen latency:  {s['avg_qwen_latency_s']} s")
    print(f"Median Qwen latency:   {s['median_qwen_latency_s']} s")
    print(f"P95 Qwen latency:      {s['p95_qwen_latency_s']} s")
    print()
    print(f"Average output tokens: {s['avg_output_tokens']}")
    print(f"Average tokens/sec:    {s['avg_tokens_per_s']}")
    print(f"Average visual tokens: {s['avg_visual_tokens']}")
    print()
    print(f"GPU peak memory:       {s['gpu_peak_allocated_gib']} GiB")
    print(f"Attention backend:     {str(s['attention_backend']).upper()}")
    print(f"Placement:             {s['placement']}")
    print("=" * 60)

    rate = s["avg_tokens_per_s"]
    print("\nWHERE THE TIME WENT")
    for k, label in [("total_render_time_s", "PDF rendering"),
                     ("total_preprocess_time_s", "preprocessing"),
                     ("total_processor_time_s", "processor/tokenisation"),
                     ("total_qwen_time_s", "Qwen generation"),
                     ("total_io_time_s", "output writing")]:
        v = s.get(k) or 0
        pct = 100 * v / max(1e-9, s["total_runtime_s"])
        print(f"  {label:<24}: {v:8.1f} s  ({pct:4.1f}%)")
    if rate is not None:
        if rate < 5:
            print(f"\n  DECODE RATE {rate} tok/s IS VERY LOW. A 27B FP8 model on an H100 should")
            print("  reach roughly 15-40 tok/s. Check PLACEMENT above: MIXED_CPU_GPU means")
            print("  weights stream over PCIe and no other setting will compensate.")
        elif s.get("pages_truncated"):
            print(f"\n  {s['pages_truncated']} page(s) hit MAX_NEW_TOKENS ({CFG.MAX_NEW_TOKENS}).")
            print("  Transcriptions are clipped -- raise it before judging OCR quality.")
        else:
            print(f"\n  Decode rate {rate} tok/s is in the expected range.")

    if s["degraded_calls"]:
        print("\nDEGRADED PAGES")
        for r in RESULTS:
            if r["ocr_status"] == "DEGRADED":
                print(f"  {r['pdf']} p{r['page']}: {r['degradation_reason']}")
                print(f"    sent {r['sent_width']}x{r['sent_height']}, "
                      f"quality={r['quality'].get('grade')}, "
                      f"preprocessing={';'.join(r['preprocessing']['applied']) or 'none'}")
                if r.get("debug"):
                    print(f"    inspect: {r['debug'].get('preprocessed_png')}")

    print("\n" + "=" * 60)
    print("TRANSCRIPTIONS (compare against the PDF and the saved images)")
    print("=" * 60)
    for r in RESULTS:
        print(f"\n--- {r['pdf']} page {r['page']}/{r['page_count']} "
              f"({r['output_tokens']} tokens, {r['generation_time_s']}s, {r['ocr_status']}) ---")
        if r.get("debug"):
            print(f"    original:     {r['debug'].get('original_png')}")
            print(f"    preprocessed: {r['debug'].get('preprocessed_png')}")
        t = r["raw_model_output"]
        print(t[:1500] + (f"\n... [{len(t)-1500} more chars in {r['txt_path']}]"
                          if len(t) > 1500 else ""))
else:
    print("no summary (DRY_RUN, or no pages were processed)")

In [ ]:
# =========================================================================
# CELL 31 — OPTIONAL: ORIGINAL vs PREPROCESSED ON ONE PAGE
# =========================================================================
# Costs exactly 2 extra Qwen calls, on ONE page, and only when you ask for it. This is how you
# find out whether the preprocessing is helping or quietly destroying detail.
def compare_preprocessing(page_index: int = 0, cfg: Config = CFG) -> pd.DataFrame:
    if cfg.DRY_RUN or not RESULTS:
        print("run the experiment first (CFG.DRY_RUN = False)")
        return pd.DataFrame()
    r = RESULTS[page_index]
    pdf_path = next((Path(x["path"]) for _, x in INVENTORY.iterrows()
                     if x["filename"] == r["pdf"]), None)
    if pdf_path is None:
        print("source PDF not found")
        return pd.DataFrame()
    doc = pymupdf.open(str(pdf_path))
    try:
        rr = render_page(doc, r["page"] - 1, cfg)
        original = rr["original"]
        quality = analyze_quality(original)
        processed, pmeta = preprocess_page(original, quality, cfg)
    finally:
        doc.close()
    plain = original.copy()
    if max(plain.size) > cfg.MAX_IMAGE_DIMENSION:
        s = cfg.MAX_IMAGE_DIMENSION / max(plain.size)
        plain = plain.resize((int(plain.width * s), int(plain.height * s)), Image.LANCZOS)

    rows = []
    for label, img, applied in [("original (resize only)", plain, ["resize"]),
                                ("preprocessed", processed, pmeta["applied"])]:
        g = qwen_generate(img, OCR_SYSTEM, OCR_USER, cfg.MAX_NEW_TOKENS, cfg)
        QWEN_CALLS["n"] += 1
        rows.append({"variant": label, "size": f"{img.width}x{img.height}",
                     "preprocessing": ";".join(applied),
                     "visual_tokens": g["visual_tokens"], "output_tokens": g["output_tokens"],
                     "generation_time_s": g["generation_time_s"],
                     "status": classify_output(g["text"], g["ids"]) or "OUTPUT_RECEIVED",
                     "text": g["text"]})
        print(f"\n===== {label} ({img.width}x{img.height}, {g['output_tokens']} tokens, "
              f"{g['generation_time_s']}s) =====")
        print(g["text"][:1200])
    df = pd.DataFrame(rows)
    df.drop(columns=["text"]).to_csv(DIRS["reports"] / "preprocessing_comparison.csv",
                                     index=False, encoding="utf-8-sig")
    print("\nCompare the two transcriptions above. More text is not automatically better:")
    print("check whether the preprocessed version preserved accents, MRZ characters,")
    print("handwriting and small print, or flattened them away.")
    return df


if CFG.COMPARE_PREPROCESSING and not CFG.DRY_RUN and RESULTS:
    COMPARISON = compare_preprocessing(0, CFG)
else:
    COMPARISON = pd.DataFrame()
    print("COMPARE_PREPROCESSING is off. Set CFG.COMPARE_PREPROCESSING = True and call "
          "compare_preprocessing(page_index) to run it on one page (2 extra Qwen calls).")

## Troubleshooting

### The sanity test failed (`!!!!!!`, empty, punctuation only)

Do not process documents. Change **one** thing at a time and re-run CELL 16:

| Order | Change | What it isolates |
|---|---|---|
| 1 | `CFG.TORCH_DTYPE = "bfloat16"` | fp16 overflow in the vision tower → NaN logits → token id 0 → `"!"` |
| 2 | `CFG.USE_FP8 = False` | FP8 block scales underflowing to zero in the patch embedding |
| 3 | `CFG.ATTENTION_BACKEND = "eager"` | an SDPA kernel producing NaN on a fully-masked row |
| 4 | `CFG.MAX_VISUAL_TOKENS = 1024` | an oversized image |

The gate prints `tokenizer.decode([0])` and the first token ids. If the ids are **all zero**, it
is NaN logits, not a prompt problem — no amount of prompt editing will fix it.

### The model reads badly but does not degenerate

This is the question the experiment exists to answer. Open three things together: the PDF page,
`04_debug/..._original.png`, `04_debug/..._preprocessed.png`, and the transcription. Then:

| What you see | Likely cause | Try |
|---|---|---|
| preprocessed looks worse than original | over-processing | `USE_THRESHOLD=False` (already default), `USE_SHARPEN=False`, `USE_DENOISE=False` |
| text tiny in the sent image | resolution lost | raise `MAX_IMAGE_DIMENSION`, raise `MAX_VISUAL_TOKENS`, check `text_height_px` in `performance_by_page.csv` |
| page sideways or upside down | rotation abstained | check `rotation_detection.margin`; lower `ROTATION_MIN_MARGIN` or rotate manually |
| transcription stops mid-page | `MAX_NEW_TOKENS` | check `pages_truncated`; raise to 768 or 1024 |
| Arabic missing, French fine | model capability | note it — this is a real finding for the next stage |
| MRZ garbled | resolution at the band | this is where a dedicated MRZ crop would help later; do not build it yet |

`performance_by_page.csv` carries the quality metrics and the preprocessing actually applied per
page, so you can correlate "bad transcription" with "low contrast" or "8 px text" rather than
guessing.

### Calls take 60–90 seconds

Read `avg_tokens_per_s` in the summary. Below 5 tok/s, look at `PLACEMENT`: `MIXED_CPU_GPU` means
part of the model is in host memory and every forward streams weights over PCIe. Lower
`RESERVE_VRAM_GIB` or free the card — nothing else will help. In the 15–40 range the rate is
healthy and the cost is volume: check `avg_visual_tokens` (prefill) and `avg_output_tokens`
(decode) to see which.

### Deliberately not here

No accuracy score — there is no ground truth, so any number would be invented. No field
extraction, no database, no cross-document matching, no MRZ reconstruction. Those come after you
can answer, from the images and the text side by side: **did it read the document?**